In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 7
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)
    
    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
17.326049206460056

Trial 1 =========================================
14.164028005598716

Trial 2 =========================================
15.904153271346072

Trial 3 =========================================
16.106929762807894

Trial 4 =========================================
14.305006163328411

Trial 5 =========================================
14.621967412962062

Trial 6 =========================================
16.96832296786531

Trial 7 =========================================
15.38890997763463

Trial 8 =========================================
14.331641842302217

Trial 9 =========================================
14.50893415644612

Trial 10 =========================================
16.088756605386827

Trial 11 =========================================
15.346325796737622



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 12 =========================================
16.176404078239216

Trial 13 =========================================
15.838346233630483

Trial 14 =========================================
18.812451199382856

Trial 15 =========================================
14.276448090704227

Trial 16 =========================================
16.238337371970257

Trial 17 =========================================
14.127466075008904

Trial 18 =========================================
16.209091072889436

Trial 19 =========================================
15.394361659354345

Trial 20 =========================================
12.799673267852961

Trial 21 =========================================
13.37971484972097



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 22 =========================================
16.22073290266168

Trial 23 =========================================
14.86435703817384

Trial 24 =========================================
14.848393865014204

Trial 25 =========================================
18.668482468659306

Trial 26 =========================================
18.779012896040324

Trial 27 =========================================
15.393567582127604



...

Trial 81 =========================================
16.12809599309037

Trial 82 =========================================
15.339401374758786

Trial 83 =========================================
15.929222010399386



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 84 =========================================
14.502603465869617

Trial 85 =========================================
14.457522351858476

Trial 86 =========================================
18.371515445015255

Trial 87 =========================================
15.8903864960496

Trial 88 =========================================
15.38476152628569

Trial 89 =========================================
16.75093142283307

Trial 90 =========================================
15.381651291701894



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 91 =========================================
16.890859070794416

Trial 92 =========================================
14.290573786206508

Trial 93 =========================================
18.497491265249465

Trial 94 =========================================
13.350645982444625

Trial 95 =========================================
17.53908015618385

Trial 96 =========================================
16.075486165854226

Trial 97 =========================================
17.473646852764844

Trial 98 =========================================
16.202893475099586

Trial 99 =========================================
15.25220424169591

[17.32604921 14.16402801 15.90415327 16.10692976 14.30500616 14.62196741
 16.96832297 15.38890998 14.33164184 14.50893416 16.08875661 15.3463258
 16.17640408 15.83834623 18.8124512  14.27644809 16.23833737 14.12746608
 16.20909107 15.39436166 12.79967327 13.37971485 16.2207329  14.86435704
 14.84839387 18.66848247 18.7790129  15.39356758 13.2628713  17.32904345

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.812451199382856
Avg = 15.497700495803501
Std = 1.3778250843919906


In [6]:
print(y_max_arr.tolist())

[17.326049206460056, 14.164028005598716, 15.904153271346072, 16.106929762807894, 14.305006163328411, 14.621967412962062, 16.96832296786531, 15.38890997763463, 14.331641842302217, 14.50893415644612, 16.088756605386827, 15.346325796737622, 16.176404078239216, 15.838346233630483, 18.812451199382856, 14.276448090704227, 16.238337371970257, 14.127466075008904, 16.209091072889436, 15.394361659354345, 12.799673267852961, 13.37971484972097, 16.22073290266168, 14.86435703817384, 14.848393865014204, 18.668482468659306, 18.779012896040324, 15.393567582127604, 13.262871296852829, 17.32904345211473, 15.395752158306008, 16.148589107026407, 14.799405700951908, 14.363391972353758, 14.260353789025732, 14.25820914361331, 16.229995543092098, 14.958842553754787, 15.384952145502863, 15.121717942083224, 14.349395915649742, 14.155466513145365, 15.013464980660713, 13.88481222184686, 16.258126231240855, 18.78167040343874, 14.446357337925658, 14.274409669545957, 17.812064413181623, 15.756056544531141, 16.078148

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-7.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)